In [ ]:
# --- repo root + config (walk parents; do not use ../..) ---
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# RQ4 — four-dataset figures (concept lane vs QA lane kept separate)

Concept: reuse stored margin-benchmark curves/AURC. QA: `m>=3` rows of
`qa_results_combined.csv`; entropy / logprob / random only. Same AURC helpers
as RQ4 Clinical Utility (trapezoid over coverage 0.1–1.0 step 0.05).

In [ ]:
from pathlib import Path
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

PROJECT_ROOT = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / "outputs" / "rq4"
ALL_FIG = PROJECT_ROOT / "outputs" / "figures" / "all_rq_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_FIG.mkdir(parents=True, exist_ok=True)

COVERAGE_GRID = np.round(np.arange(1.00, 0.09, -0.05), 2)
N_RANDOM = 20
RNG = np.random.default_rng(42)

MODEL_ORDER_CONCEPT = [
    "BERT-base", "BioBERT", "PubMedBERT", "FLAN-T5-base",
    "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
]
MODEL_ORDER_QA = [
    "FLAN-T5-base", "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
]
SHORT = {
    "Mistral-7B-Instruct-v0.1": "Mistral-7B",
    "Llama3-OpenBioLLM-8B": "OpenBioLLM-8B",
    "OpenBioLLM-8B": "OpenBioLLM-8B",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "FLAN-T5-base": "FLAN-T5",
}

# Canonical displayed names. CSV column 'combined' is the 2-signal ranker.
SIGNAL_LABELS = {
    "entropy":     "Semantic entropy",
    "confidence":  "Mapping confidence",
    "margin":      "UMLS margin",
    "random":      "Random (no ranking)",
    "combined_2":  "Entropy, then confidence if tied",
    "combined_3":  "Mean rank of entropy + confidence + margin",
}
SPECS = {
    "entropy":     ("-",  "#C44E52", SIGNAL_LABELS["entropy"]),
    "confidence":  ("-.", "#4C72B0", SIGNAL_LABELS["confidence"]),
    "margin":      ("-",  "#E69F00", SIGNAL_LABELS["margin"]),
    "random":      ("--", "#55A868", SIGNAL_LABELS["random"]),
    "combined":    ("-",  "#56B4E9", SIGNAL_LABELS["combined_2"]),
    "combined_3":  (":",  "#882255", SIGNAL_LABELS["combined_3"]),
}
CONCEPT_SIGNALS = ["entropy", "confidence", "random", "margin", "combined", "combined_3"]
QA_SIGNALS = ["entropy", "confidence", "random"]

def selective_curve(y, score_order, coverages=COVERAGE_GRID):
    ranked_y = y[score_order]
    n = len(y)
    rows = []
    for cov in coverages:
        k = max(1, int(np.ceil(float(cov) * n)))
        acc = float(np.mean(ranked_y[:k]))
        rows.append({
            "coverage": float(cov), "n_keep": int(k), "n_total": int(n),
            "selective_accuracy": acc, "risk": 1.0 - acc,
        })
    return rows

def aurc_from_curve(coverages, risks):
    c = np.asarray(coverages, dtype=float)
    r = np.asarray(risks, dtype=float)
    order = np.argsort(c)
    trapz = getattr(np, "trapezoid", None) or np.trapz
    return float(trapz(r[order], c[order]))

print("helpers ready")


In [ ]:
# Concept lane: on-disk curves + AURC (do not recompute)
mm_curves = pd.read_csv(OUT_DIR / "rq4_risk_coverage_margin_medmentions.csv")
cad_curves = pd.read_csv(OUT_DIR / "rq4_risk_coverage_margin_cadec.csv")
concept_aurc = pd.read_csv(OUT_DIR / "rq4_aurc_margin_benchmark.csv")
concept_curves = pd.concat([mm_curves, cad_curves], ignore_index=True)
concept_curves = concept_curves[concept_curves["signal"].isin(CONCEPT_SIGNALS)].copy()
print("concept curves", concept_curves.groupby(["dataset", "signal"]).size().to_string())

# QA lane: m>=3; correctness = `correct` (already computed); entropy = norm_entropy
# (QA notebook ranked on normalised entropy; same 0–1 scale as the concept lane).
qa = pd.read_csv(PROJECT_ROOT / "outputs/qa/qa_results_combined.csv")
qa = qa[qa["m"] >= 3].copy()
qa["y"] = qa["correct"].map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0})
qa["y"] = pd.to_numeric(qa["y"], errors="coerce")
qa["entropy"] = pd.to_numeric(qa["norm_entropy"], errors="coerce")
qa["confidence"] = pd.to_numeric(qa["confidence"], errors="coerce")
qa = qa.dropna(subset=["y", "entropy", "confidence"])
qa["dataset_label"] = qa["dataset"].map({"bioasq": "BioASQ", "squad2": "SQuAD"})
print("QA kept", len(qa), qa.groupby(["dataset_label", "model"]).size().to_string())

qa_curve_rows, qa_aurc_rows = [], []
for (ds, model), g in qa.groupby(["dataset_label", "model"], sort=False):
    g = g.reset_index(drop=True)
    y = g["y"].to_numpy(dtype=float)
    h = g["entropy"].to_numpy(dtype=float)
    conf = g["confidence"].to_numpy(dtype=float)
    n = len(y)
    orders = {
        "entropy": np.argsort(h, kind="mergesort"),          # high H = risky
        "confidence": np.argsort(-conf, kind="mergesort"),   # less-negative logprob = safer
    }
    curves = {}
    for sig, order in orders.items():
        curve = selective_curve(y, order)
        for r in curve:
            r.update({"dataset": ds, "model": model, "signal": sig, "n": n})
            qa_curve_rows.append(r)
        curves[sig] = curve
    acc_by_cov = {float(c): [] for c in COVERAGE_GRID}
    for _ in range(N_RANDOM):
        perm = RNG.permutation(n)
        ranked_y = y[perm]
        for cov in COVERAGE_GRID:
            k = max(1, int(np.ceil(float(cov) * n)))
            acc_by_cov[float(cov)].append(float(np.mean(ranked_y[:k])))
    rand_curve = []
    for cov in COVERAGE_GRID:
        acc = float(np.mean(acc_by_cov[float(cov)]))
        row = {
            "dataset": ds, "model": model, "signal": "random",
            "coverage": float(cov), "n": n,
            "selective_accuracy": acc, "risk": 1.0 - acc,
        }
        qa_curve_rows.append(row)
        rand_curve.append(row)
    curves["random"] = rand_curve
    for sig in QA_SIGNALS:
        qa_aurc_rows.append({
            "dataset": ds, "model": model, "n": n, "signal": sig,
            "AURC": aurc_from_curve(
                [r["coverage"] for r in curves[sig]],
                [r["risk"] for r in curves[sig]],
            ),
        })

qa_curves = pd.DataFrame(qa_curve_rows)
qa_aurc = pd.DataFrame(qa_aurc_rows)
print("\nQA AURC (trapezoid 0.1–1.0; not the QA-notebook mean-of-prefixes):")
print(qa_aurc.pivot(index=["dataset", "model", "n"], columns="signal", values="AURC").round(4).to_string())


In [ ]:
# FIGURE 1 — risk–coverage, 4 dataset sections, lanes kept separate
def _n_lookup(curves, model):
    sub = curves[curves["model"] == model]
    if "n" in sub.columns and sub["n"].notna().any():
        return int(sub["n"].dropna().iloc[0])
    return None

def _plot_model(ax, curves, model, signals, lane, n_note=None, small=False):
    sub = curves[curves["model"] == model]
    for sig in signals:
        style, color, label = SPECS[sig]
        s = sub[sub["signal"] == sig].sort_values("coverage")
        if s.empty:
            continue
        ax.plot(
            s["coverage"], s["risk"],
            linestyle=style, marker="o", markersize=2.5, linewidth=1.5,
            color=color, label=label,
        )
    short = SHORT.get(model, model)
    extra = ""
    if small and n_note is not None:
        extra = f" (n={n_note}, SMALL)"
    elif n_note is not None:
        extra = f" (n={n_note})"
    ax.set_title(f"{lane}\n{short}{extra}", fontsize=8.5, pad=4)
    ax.set_xlabel("Coverage")
    ax.set_ylabel("Risk")
    ax.set_xlim(0.08, 1.02)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelbottom=True, labelsize=8)

fig = plt.figure(figsize=(18.0, 21.5))
outer = gridspec.GridSpec(4, 1, figure=fig, height_ratios=[1.05, 2.15, 1.05, 1.05], hspace=0.42)

# --- MedMentions (concept, 5 generatives) ---
gs0 = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=outer[0], wspace=0.32)
mm = concept_curves[concept_curves["dataset"] == "MedMentions"]
_ENCODERS = {"BERT-base", "BioBERT", "PubMedBERT"}
mm_models = [m for m in MODEL_ORDER_CONCEPT if m in set(mm["model"]) and m not in _ENCODERS]
for i, model in enumerate(mm_models):
    ax = fig.add_subplot(gs0[0, i])
    _plot_model(ax, mm, model, CONCEPT_SIGNALS, "concept-level entropy",
                n_note=_n_lookup(mm, model))
    if i == 0:
        ax.text(
            -0.15, 1.28, "MedMentions — concept-level entropy  (encoders excluded: direct-CUI, margin undefined)",
            transform=ax.transAxes, fontsize=11, fontweight="bold", clip_on=False,
        )

# --- CADEC (concept, 8 models) ---
gs1 = gridspec.GridSpecFromSubplotSpec(2, 4, subplot_spec=outer[1], wspace=0.32, hspace=0.55)
cad = concept_curves[concept_curves["dataset"] == "CADEC"]
cad_models = [m for m in MODEL_ORDER_CONCEPT if m in set(cad["model"])]
for i, model in enumerate(cad_models):
    ax = fig.add_subplot(gs1[i // 4, i % 4])
    n = _n_lookup(cad, model)
    small = (model == "Llama3-OpenBioLLM-8B")
    _plot_model(ax, cad, model, CONCEPT_SIGNALS, "concept-level entropy",
                n_note=n, small=small)
    if i == 0:
        ax.text(
            -0.15, 1.32, "CADEC — concept-level entropy  (OpenBioLLM flagged n=233; margin not imputed)",
            transform=ax.transAxes, fontsize=11, fontweight="bold", clip_on=False,
        )

# --- BioASQ (QA, 5 models) ---
gs2 = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=outer[2], wspace=0.32)
bio = qa_curves[qa_curves["dataset"] == "BioASQ"]
bio_models = [m for m in MODEL_ORDER_QA if m in set(bio["model"])]
for i, model in enumerate(bio_models):
    ax = fig.add_subplot(gs2[0, i])
    _plot_model(ax, bio, model, QA_SIGNALS, "answer-level entropy",
                n_note=392)
    if i == 0:
        ax.text(
            -0.15, 1.28, "BioASQ — answer-level entropy  (m≥3, n=392 per model; "
            f"no {SIGNAL_LABELS['margin']} / no {SIGNAL_LABELS['combined_3']})",
            transform=ax.transAxes, fontsize=11, fontweight="bold", clip_on=False,
        )

# --- SQuAD (QA, 5 models, thin n=139) ---
gs3 = gridspec.GridSpecFromSubplotSpec(1, 5, subplot_spec=outer[3], wspace=0.32)
sq = qa_curves[qa_curves["dataset"] == "SQuAD"]
sq_models = [m for m in MODEL_ORDER_QA if m in set(sq["model"])]
for i, model in enumerate(sq_models):
    ax = fig.add_subplot(gs3[0, i])
    _plot_model(ax, sq, model, QA_SIGNALS, "answer-level entropy",
                n_note=139, small=True)
    if i == 0:
        ax.text(
            -0.15, 1.28, "SQuAD — answer-level entropy  (m≥3, n=139 per model, SMALL; "
            f"no {SIGNAL_LABELS['margin']} / no {SIGNAL_LABELS['combined_3']})",
            transform=ax.transAxes, fontsize=11, fontweight="bold", clip_on=False,
        )

legend_handles = [
    Line2D([0], [0], color=SPECS[s][1], linestyle=SPECS[s][0], marker="o",
           markersize=4, linewidth=1.6, label=SPECS[s][2])
    for s in CONCEPT_SIGNALS
]
fig.legend(
    legend_handles, [SPECS[s][2] for s in CONCEPT_SIGNALS],
    loc="upper center", ncol=3, frameon=False, fontsize=10,
    bbox_to_anchor=(0.5, 0.995),
)
fig.suptitle(
    "RQ4 risk–coverage  ·  two lanes kept separate\n"
    "risk = 1 − selective accuracy; lower is better   ·   "
    f"{SIGNAL_LABELS['margin']} / {SIGNAL_LABELS['combined_3']} exist only on the concept lane",
    y=1.015, fontsize=13,
)
fig.text(
    0.5, -0.01,
    "confidence = SapBERT cosine (concept) vs sequence logprob (QA);  "
    "correctness = CUI match vs answer match;  "
    "curves comparable within a lane, not across lanes.",
    ha="center", va="top", fontsize=9, color="#333333",
)
plt.close(fig)
print("Skipping 4-dataset combined RC PNG; six-file regen is cell 9833ab53.")


In [ ]:
# FIGURE 2 — AURC bars, 4 stacked panels (lower = better)
concept_wide = (
    concept_aurc[concept_aurc["signal"].isin(CONCEPT_SIGNALS)]
    .copy()
)

def _models_for(ds, lane):
    if lane == "concept":
        have = set(concept_wide.loc[concept_wide["dataset"] == ds, "model"])
        models = [m for m in MODEL_ORDER_CONCEPT if m in have]
        if ds == "MedMentions":
            models = [m for m in models if m not in {"BERT-base", "BioBERT", "PubMedBERT"}]
        return models
    have = set(qa_aurc.loc[qa_aurc["dataset"] == ds, "model"])
    return [m for m in MODEL_ORDER_QA if m in have]

def _aurc_vec(ds, models, signal, lane):
    if lane == "concept":
        sub = concept_wide[(concept_wide["dataset"] == ds) & (concept_wide["signal"] == signal)]
        mcol = "model"
    else:
        sub = qa_aurc[(qa_aurc["dataset"] == ds) & (qa_aurc["signal"] == signal)]
        mcol = "model"
    lookup = dict(zip(sub[mcol], sub["AURC"]))
    return [lookup.get(m, np.nan) for m in models]

panels = [
    ("MedMentions", "concept", CONCEPT_SIGNALS,
     "MedMentions — concept-level entropy  (generatives only)"),
    ("CADEC", "concept", CONCEPT_SIGNALS,
     "CADEC — concept-level entropy  (OpenBioLLM n=233 SMALL)"),
    ("BioASQ", "qa", QA_SIGNALS,
     "BioASQ — answer-level entropy  (m≥3, n=392; no margin)"),
    ("SQuAD", "qa", QA_SIGNALS,
     "SQuAD — answer-level entropy  (m≥3, n=139 SMALL; no margin)"),
]

fig, axes = plt.subplots(4, 1, figsize=(13.5, 16.5))
bar_w = {"concept": 0.15, "qa": 0.22}

for ax, (ds, lane, signals, title) in zip(axes, panels):
    models = _models_for(ds, lane)
    x = np.arange(len(models))
    w = bar_w[lane]
    n_sig = len(signals)
    offsets = (np.arange(n_sig) - (n_sig - 1) / 2.0) * w
    for off, sig in zip(offsets, signals):
        vals = _aurc_vec(ds, models, sig, lane)
        ax.bar(
            x + off, vals, width=w * 0.95,
            color=SPECS[sig][1], label=SPECS[sig][2], zorder=3,
            edgecolor="none",
        )
    labels = []
    for m in models:
        lab = SHORT.get(m, m)
        if ds == "CADEC" and m == "Llama3-OpenBioLLM-8B":
            lab = f"{lab}\nn=233"
        labels.append(lab)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8.5)
    ax.set_ylabel("AURC")
    ax.set_title(title, loc="left", fontsize=11)
    ax.grid(True, axis="y", alpha=0.35, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(labelbottom=True)

handles = [Patch(facecolor=SPECS[s][1], label=SPECS[s][2]) for s in CONCEPT_SIGNALS]
fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False,
           bbox_to_anchor=(0.5, 1.01), fontsize=10)
fig.suptitle(
    "RQ4 AURC  ·  lower is better  ·  two lanes kept separate",
    y=1.035, fontsize=13,
)
fig.text(
    0.5, -0.01,
    "confidence = SapBERT cosine (concept) vs sequence logprob (QA);  "
    "correctness = CUI match vs answer match;  "
    "AURC comparable within a lane, not across lanes.  "
    f"{SIGNAL_LABELS['margin']} / {SIGNAL_LABELS['combined_3']}: concept lane only.",
    ha="center", va="top", fontsize=9, color="#333333",
)
fig.tight_layout(rect=(0, 0.02, 1, 0.98))
plt.close(fig)
print("Skipping 4-dataset combined AURC PNG; six-file regen is cell 9833ab53.")


## Per-dataset grids + two AURC figures (real files only)

- One risk–coverage PNG per dataset.
- MedMentions/CADEC: all 8 models. Encoder UMLS margin uses the **input mention span** and the predicted CUI (not the gold label).
- BioASQ and SQuAD2 omit UMLS margin and the 3-signal combiner (QA lane has no UMLS candidate margin).
- AURC: one figure for MedMentions+CADEC, one for BioASQ+SQuAD2.
- PNG regen is the **display-only** cell at the end (loads stored CSVs; does not recompute scores).


In [ ]:
# Per-dataset risk-coverage (one PNG each) + two AURC figures.
# Every line/bar comes from an on-disk CSV or from selective_curve on qa_results_combined.
# If a (dataset, model, signal) is missing, it is skipped — never filled in.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

PROJECT_ROOT = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / "outputs" / "rq4"
ALL_FIG = PROJECT_ROOT / "outputs" / "figures" / "all_rq_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_FIG.mkdir(parents=True, exist_ok=True)

COVERAGE_GRID = np.round(np.arange(1.00, 0.09, -0.05), 2)
N_RANDOM = 20
RNG = np.random.default_rng(42)

MODEL_ORDER_8 = [
    "BERT-base", "BioBERT", "PubMedBERT", "FLAN-T5-base",
    "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
]
MODEL_ORDER_QA = [
    "FLAN-T5-base", "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
]
ENCODERS = {"BERT-base", "BioBERT", "PubMedBERT"}
SHORT = {
    "Mistral-7B-Instruct-v0.1": "Mistral-7B",
    "Llama3-OpenBioLLM-8B": "OpenBioLLM-8B",
    "OpenBioLLM-8B": "OpenBioLLM-8B",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "FLAN-T5-base": "FLAN-T5",
}
SIGNAL_LABELS = {
    "entropy":     "Semantic entropy",
    "confidence":  "Mapping confidence",
    "margin":      "UMLS margin",
    "random":      "Random (no ranking)",
    "combined_2":  "Entropy, then confidence if tied",
    "combined_3":  "Mean rank of entropy + confidence + margin",
}
SPECS = {
    "entropy":     ("-",  "#C44E52", SIGNAL_LABELS["entropy"]),
    "confidence":  ("-.", "#4C72B0", SIGNAL_LABELS["confidence"]),
    "margin":      ("-",  "#E69F00", SIGNAL_LABELS["margin"]),
    "random":      ("--", "#55A868", SIGNAL_LABELS["random"]),
    "combined":    ("-",  "#56B4E9", SIGNAL_LABELS["combined_2"]),
    "combined_3":  (":",  "#882255", SIGNAL_LABELS["combined_3"]),
}
CONCEPT_SIGNALS = ["entropy", "confidence", "random", "margin", "combined", "combined_3"]
ALL_SIGNALS = CONCEPT_SIGNALS  # same legend / bar slots on every figure
QA_SIGNALS_REAL = ["entropy", "confidence", "random"]  # only these exist in QA files


def lookup_rows(df, model):
    """Align concept-lane names with QA file names. Never invent a row."""
    aliases = {model}
    if model == "Llama3-OpenBioLLM-8B":
        aliases.add("OpenBioLLM-8B")
    if model == "OpenBioLLM-8B":
        aliases.add("Llama3-OpenBioLLM-8B")
    return df[df["model"].isin(aliases)]


def selective_curve(y, score_order, coverages=COVERAGE_GRID):
    ranked_y = y[score_order]
    n = len(y)
    rows = []
    for cov in coverages:
        k = max(1, int(np.ceil(float(cov) * n)))
        acc = float(np.mean(ranked_y[:k]))
        rows.append({
            "coverage": float(cov), "n_keep": int(k), "n_total": int(n),
            "selective_accuracy": acc, "risk": 1.0 - acc,
        })
    return rows


def aurc_from_curve(coverages, risks):
    c = np.asarray(coverages, dtype=float)
    r = np.asarray(risks, dtype=float)
    order = np.argsort(c)
    trapz = getattr(np, "trapezoid", None) or np.trapz
    return float(trapz(r[order], c[order]))


# ---------- load REAL concept curves ----------
mm_gen = pd.read_csv(OUT_DIR / "rq4_risk_coverage_margin_medmentions.csv")
mm_gen = mm_gen[mm_gen["signal"].isin(CONCEPT_SIGNALS)].copy()
mm_gen["source"] = "margin_benchmark"

mm_old = pd.read_csv(PROJECT_ROOT / "outputs/rq1/rq4_risk_coverage_medmentions.csv")
# signal is a column of names, not a numeric series named mapping_confidence
mm_old["signal"] = mm_old["signal"].replace({"mapping_confidence": "confidence"})
mm_old = mm_old[mm_old["model"].isin(ENCODERS) & mm_old["signal"].isin(["entropy", "confidence", "random"])].copy()
assert not mm_old.empty, "encoder curves missing after signal rename — check rq4_risk_coverage_medmentions.csv"
mm_old["n"] = 491
mm_old["small_n"] = False
mm_old["source"] = "rq4_risk_coverage_medmentions.csv"
mm_curves = pd.concat([mm_gen, mm_old], ignore_index=True)

cad_curves = pd.read_csv(OUT_DIR / "rq4_risk_coverage_margin_cadec.csv")
cad_curves = cad_curves[cad_curves["signal"].isin(CONCEPT_SIGNALS)].copy()
cad_curves["source"] = "margin_benchmark"

print("MM models in figure:", sorted(mm_curves.model.unique()))
print("MM encoder signals (must NOT include margin):",
      sorted(mm_curves.loc[mm_curves.model.isin(ENCODERS), "signal"].unique()))
print("CADEC models:", sorted(cad_curves.model.unique()))

# ---------- load REAL concept AURC ----------
mm_aurc_gen = pd.read_csv(OUT_DIR / "rq4_aurc_margin_benchmark.csv")
mm_aurc_gen = mm_aurc_gen[(mm_aurc_gen.dataset == "MedMentions") & mm_aurc_gen.signal.isin(CONCEPT_SIGNALS)]
# Encoder AURC from the same stored curves (same trapezoid as the rest of RQ4)
enc_rows = []
for (model, sig), s in mm_old.groupby(["model", "signal"]):
    s = s.sort_values("coverage")
    enc_rows.append({
        "dataset": "MedMentions", "model": model, "n": 491, "signal": sig,
        "AURC": aurc_from_curve(s["coverage"], s["risk"]),
        "source": "rq4_risk_coverage_medmentions.csv + trapezoid",
    })
mm_enc_sum = pd.read_csv(PROJECT_ROOT / "outputs/rq1/rq4_aurc_summary.csv")
mm_enc_sum = mm_enc_sum[mm_enc_sum.model.isin(ENCODERS)]
print("MM encoder AURC from curves vs rq4_aurc_summary.csv (must match):")
for row in enc_rows:
    col = {"entropy": "AURC_entropy", "confidence": "AURC_confidence", "random": "AURC_random"}[row["signal"]]
    file_val = float(mm_enc_sum.loc[mm_enc_sum.model == row["model"], col].iloc[0])
    delta = abs(row["AURC"] - file_val)
    print(f"  {row['model']:12s} {row['signal']:12s} curve={row['AURC']:.6f} file={file_val:.6f} |d|={delta:.2e}")
    if delta > 1e-4:
        raise AssertionError(f"encoder AURC mismatch for {row['model']} {row['signal']}")
mm_aurc = pd.concat([
    mm_aurc_gen.assign(source="margin_benchmark")[["dataset", "model", "n", "signal", "AURC", "source"]],
    pd.DataFrame(enc_rows),
], ignore_index=True)

cad_aurc = pd.read_csv(OUT_DIR / "rq4_aurc_margin_benchmark.csv")
cad_aurc = cad_aurc[(cad_aurc.dataset == "CADEC") & cad_aurc.signal.isin(CONCEPT_SIGNALS)].copy()
cad_aurc["source"] = "margin_benchmark"

# ---------- QA from qa_results_combined (m>=3), same helpers ----------
qa = pd.read_csv(PROJECT_ROOT / "outputs/qa/qa_results_combined.csv")
qa = qa[qa["m"] >= 3].copy()
qa["y"] = pd.to_numeric(qa["correct"].map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}), errors="coerce")
qa["entropy"] = pd.to_numeric(qa["norm_entropy"], errors="coerce")
qa["confidence"] = pd.to_numeric(qa["confidence"], errors="coerce")
qa = qa.dropna(subset=["y", "entropy", "confidence"])
qa["dataset_label"] = qa["dataset"].map({"bioasq": "BioASQ", "squad2": "SQuAD2"})
assert set(qa.model.unique()) <= set(MODEL_ORDER_QA)
assert not set(qa.model) & ENCODERS, "QA file has encoder rows — unexpected"

qa_curve_rows, qa_aurc_rows = [], []
for (ds, model), g in qa.groupby(["dataset_label", "model"], sort=False):
    g = g.reset_index(drop=True)
    y = g["y"].to_numpy(dtype=float)
    h = g["entropy"].to_numpy(dtype=float)
    conf = g["confidence"].to_numpy(dtype=float)
    n = len(y)
    orders = {
        "entropy": np.argsort(h, kind="mergesort"),
        "confidence": np.argsort(-conf, kind="mergesort"),
    }
    curves = {}
    for sig, order in orders.items():
        curve = selective_curve(y, order)
        for r in curve:
            r.update({"dataset": ds, "model": model, "signal": sig, "n": n})
            qa_curve_rows.append(r)
        curves[sig] = curve
    acc_by_cov = {float(c): [] for c in COVERAGE_GRID}
    for _ in range(N_RANDOM):
        perm = RNG.permutation(n)
        ranked_y = y[perm]
        for cov in COVERAGE_GRID:
            k = max(1, int(np.ceil(float(cov) * n)))
            acc_by_cov[float(cov)].append(float(np.mean(ranked_y[:k])))
    rand_curve = []
    for cov in COVERAGE_GRID:
        acc = float(np.mean(acc_by_cov[float(cov)]))
        row = {"dataset": ds, "model": model, "signal": "random",
               "coverage": float(cov), "n": n,
               "selective_accuracy": acc, "risk": 1.0 - acc}
        qa_curve_rows.append(row)
        rand_curve.append(row)
    curves["random"] = rand_curve
    for sig in QA_SIGNALS_REAL:
        qa_aurc_rows.append({
            "dataset": ds, "model": model, "n": n, "signal": sig,
            "AURC": aurc_from_curve([r["coverage"] for r in curves[sig]],
                                    [r["risk"] for r in curves[sig]]),
            "source": "qa_results_combined.csv m>=3 trapezoid",
        })

qa_curves = pd.DataFrame(qa_curve_rows)
qa_aurc = pd.DataFrame(qa_aurc_rows)
print("QA models (not 8 — encoders never run):", sorted(qa_curves.model.unique()))
print(qa_aurc.pivot(index=["dataset", "model", "n"], columns="signal", values="AURC").round(4).to_string())


def plot_rc_grid(curves, models, signals, title, outfile, lane, n_override=None, small_models=None):
    small_models = small_models or set()
    n_models = len(models)
    ncols = 4
    nrows = int(np.ceil(n_models / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.15 * ncols, 3.55 * nrows),
                             sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, model in zip(axes, models):
        sub = lookup_rows(curves, model)
        plotted = []
        for sig in signals:
            s = sub[sub["signal"] == sig].sort_values("coverage")
            if s.empty:
                continue
            style, color, _label = SPECS[sig]
            ax.plot(s["coverage"], s["risk"], linestyle=style, marker="o",
                    markersize=3, linewidth=1.55, color=color)
            plotted.append(sig)
        n = n_override
        if n is None and "n" in sub.columns and sub["n"].notna().any():
            n = int(sub["n"].dropna().iloc[0])
        short = SHORT.get(model, model)
        if not plotted:
            ax.set_title(f"{lane}\n{short}\nnot run — no data", fontsize=9, color="#666666")
        else:
            flag = ""
            if model in small_models or (n is not None and n < 250):
                flag = f" (n={n}, SMALL)" if n is not None else " (SMALL)"
            elif n is not None:
                flag = f" (n={n})"
            missing = [SPECS[s][2] for s in signals if s not in plotted]
            miss_txt = f"\nn/a: {', '.join(missing)}" if missing else ""
            ax.set_title(f"{lane}\n{short}{flag}{miss_txt}", fontsize=9)
        ax.set_xlabel("Coverage")
        ax.set_ylabel("Risk")
        ax.set_xlim(0.08, 1.02)
        ax.set_ylim(0.0, 1.02)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelbottom=True)
    for ax in axes[n_models:]:
        ax.axis("off")
    handles = [Line2D([0], [0], color=SPECS[s][1], linestyle=SPECS[s][0],
                      marker="o", markersize=4, linewidth=1.6, label=SPECS[s][2])
               for s in signals]
    handles.append(Line2D([0], [0], color="#888888", linestyle="None",
                          marker="x", markersize=6, label="n/a (not defined)"))
    fig.legend(handles, [h.get_label() for h in handles], loc="upper center",
               ncol=len(handles), frameon=False, bbox_to_anchor=(0.5, 1.05), fontsize=9)
    fig.suptitle(title + "\nrisk = 1 − selective accuracy; lower is better", y=1.10, fontsize=12)
    fig.tight_layout()
    fig.savefig(outfile, dpi=150, bbox_inches="tight")
    fig.savefig(ALL_FIG / outfile.name.replace("rq4_", "RQ4_"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved", outfile)


print("Skipping PNG writes here. Display-only regen is the next cell (stored CSVs only).")


def plot_aurc_two_panels(panel_specs, outfile, footnote):
    fig, axes = plt.subplots(2, 1, figsize=(13.2, 9.2), sharey=True)
    signals = ALL_SIGNALS
    for ax, spec in zip(axes, panel_specs):
        ds, models, aurc_df, title = spec
        x = np.arange(len(models))
        w = 0.14
        offsets = (np.arange(len(signals)) - (len(signals) - 1) / 2.0) * w
        for off, sig in zip(offsets, signals):
            vals, present = [], []
            for m in models:
                hit = lookup_rows(aurc_df, m)
                hit = hit[hit["signal"] == sig]
                if len(hit):
                    vals.append(float(hit["AURC"].iloc[0]))
                    present.append(True)
                else:
                    vals.append(np.nan)
                    present.append(False)
            ax.bar(x + off, vals, width=w * 0.92, color=SPECS[sig][1],
                   label=SPECS[sig][2], zorder=3, edgecolor="none")
            for xi, ok in zip(x + off, present):
                if not ok:
                    ax.plot(xi, 0.04, marker="x", color=SPECS[sig][1],
                            markersize=7, zorder=4, clip_on=False)
        labels = []
        for m in models:
            lab = SHORT.get(m, m)
            n_hit = lookup_rows(aurc_df, m)["n"].dropna() if "n" in aurc_df.columns else pd.Series(dtype=float)
            if len(n_hit):
                lab = f"{lab}\nn={int(n_hit.iloc[0])}"
            else:
                lab = f"{lab}\nnot run"
            labels.append(lab)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8.5)
        ax.set_ylabel("AURC")
        ax.set_ylim(0.0, 1.02)
        ax.set_title(title, loc="left", fontsize=11)
        ax.grid(True, axis="y", alpha=0.35, zorder=0)
        ax.set_axisbelow(True)
    handles = [Patch(facecolor=SPECS[s][1], label=SPECS[s][2]) for s in signals]
    handles.append(Line2D([0], [0], color="#888888", linestyle="None",
                          marker="x", markersize=7, label="n/a (not defined)"))
    fig.legend(handles=handles, loc="upper center", ncol=len(handles),
               frameon=False, bbox_to_anchor=(0.5, 1.03), fontsize=9)
    fig.suptitle("RQ4 AURC  ·  lower is better  ·  same 8 models and 5 signals on both panels",
                 y=1.055, fontsize=13)
    fig.text(0.5, -0.02, footnote, ha="center", va="top", fontsize=8.5, color="#333333")
    fig.tight_layout(rect=(0, 0.03, 1, 0.97))
    fig.savefig(outfile, dpi=150, bbox_inches="tight")
    fig.savefig(ALL_FIG / outfile.name.replace("rq4_", "RQ4_"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved", outfile)


print("Skipping PNG writes here. Display-only regen is the next cell (stored CSVs only).")

print("\n=== SOURCE TRACE (plotted AURC must exist in these files) ===")
print("MM encoder AURC from outputs/rq1/rq4_aurc_summary.csv:")
print(mm_aurc[mm_aurc.model.isin(ENCODERS)].round(4).to_string(index=False))
print("\nMM generative AURC from umls margin benchmark:")
print(mm_aurc[~mm_aurc.model.isin(ENCODERS)].pivot(index="model", columns="signal", values="AURC").round(4).to_string())
print("\nCADEC AURC from umls margin benchmark:")
print(cad_aurc.pivot(index=["model", "n"], columns="signal", values="AURC").round(4).to_string())
print("\nQA AURC from qa_results_combined m>=3:")
print(qa_aurc.pivot(index=["dataset", "model", "n"], columns="signal", values="AURC").round(4).to_string())


## Display-only: six RQ4 figures (no score recompute)

Loads stored curve/AURC CSVs only. Writes six PNGs at 200 dpi into
`outputs/rq4/present/figures/`:
- concept risk–coverage: MedMentions and CADEC (8 models, 6 signals)
- QA risk–coverage: BioASQ and SQuAD (5 generatives, 4 signals; no margin / no 3-signal)
- AURC: concept lane (MM+CADEC) and QA lane (BioASQ+SQuAD)

Every legend uses SIGNAL_LABELS. MedMentions encoders are included (confidence ≡ 1.0;
margin = input-mention-span gap).


In [ ]:
# DISPLAY ONLY. Six RQ4 figures from stored CSVs. Do not recompute scores.
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

PROJECT_ROOT = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / "outputs" / "rq4"
ALL_FIG = PROJECT_ROOT / "outputs" / "figures" / "all_rq_figures"
PRESENT = OUT_DIR / "present" / "figures"
ALL_FIG.mkdir(exist_ok=True)
PRESENT.mkdir(parents=True, exist_ok=True)
DPI = 200

CSV_PATHS = [
    OUT_DIR / "rq4_risk_coverage_margin_medmentions.csv",
    OUT_DIR / "rq4_risk_coverage_margin_cadec.csv",
    OUT_DIR / "rq4_risk_coverage_margin_qa.csv",
    OUT_DIR / "rq4_aurc_margin_benchmark.csv",
    OUT_DIR / "rq4_aurc_margin_qa.csv",
]


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


hashes_before = {p.name: file_sha256(p) for p in CSV_PATHS}

SIGNAL_LABELS = {
    "entropy":     "Semantic entropy",
    "confidence":  "Mapping confidence",
    "margin":      "UMLS margin",
    "random":      "Random (no ranking)",
    "combined_2":  "Entropy, then confidence if tied",
    "combined_3":  "Mean rank of entropy + confidence + margin",
}

# Same color/linestyle for a signal on every figure.
STYLES = {
    "entropy":     ("-",  "#C44E52"),
    "confidence":  ("-.", "#4C72B0"),
    "margin":      ("-",  "#E69F00"),
    "random":      ("--", "#55A868"),
    "combined":    ("-",  "#56B4E9"),
    "combined_3":  (":",  "#882255"),
}
CONCEPT_SIGNALS = ["entropy", "confidence", "margin", "random", "combined", "combined_3"]
QA_SIGNALS = ["entropy", "confidence", "random", "combined"]


def signal_label(sig):
    if sig == "combined":
        return SIGNAL_LABELS["combined_2"]
    return SIGNAL_LABELS[sig]


assert all(signal_label(s) == SIGNAL_LABELS[k] for s, k in [
    ("entropy", "entropy"), ("confidence", "confidence"), ("margin", "margin"),
    ("random", "random"), ("combined", "combined_2"), ("combined_3", "combined_3"),
])
assert "combined_3" not in {signal_label(s) for s in CONCEPT_SIGNALS}
assert "2-signal" not in {signal_label(s) for s in CONCEPT_SIGNALS}
print("Legend labels (concept, 6):", [signal_label(s) for s in CONCEPT_SIGNALS])
print("Legend labels (QA, 4):", [signal_label(s) for s in QA_SIGNALS])

MODEL_ORDER_8 = [
    "BERT-base", "BioBERT", "PubMedBERT", "FLAN-T5-base",
    "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
]
MODEL_ORDER_QA = [
    "FLAN-T5-base", "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
]
SHORT = {
    "Mistral-7B-Instruct-v0.1": "Mistral-7B",
    "Llama3-OpenBioLLM-8B": "OpenBioLLM-8B",
    "OpenBioLLM-8B": "OpenBioLLM-8B",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "FLAN-T5-base": "FLAN-T5",
}

MM_ENCODER_FOOTNOTE = (
    "MedMentions encoders are direct-CUI: mapping confidence is constant 1.0 "
    "(degenerate ranker) and UMLS margin is the input-mention-span gap (mention difficulty), "
    "not a model-uncertainty axis."
)
QA_FOOTNOTE = (
    "Answer-level NLI entropy; UMLS margin is a concept-lane metric "
    "and is not computed here."
)


def lookup_rows(df, model):
    aliases = {model}
    if model == "Llama3-OpenBioLLM-8B":
        aliases.add("OpenBioLLM-8B")
    if model == "OpenBioLLM-8B":
        aliases.add("Llama3-OpenBioLLM-8B")
    return df[df["model"].isin(aliases)]


mm_curves = pd.read_csv(OUT_DIR / "rq4_risk_coverage_margin_medmentions.csv")
cad_curves = pd.read_csv(OUT_DIR / "rq4_risk_coverage_margin_cadec.csv")
qa_curves = pd.read_csv(OUT_DIR / "rq4_risk_coverage_margin_qa.csv")
mm_aurc = pd.read_csv(OUT_DIR / "rq4_aurc_margin_benchmark.csv")
mm_aurc = mm_aurc[mm_aurc["dataset"] == "MedMentions"].copy()
cad_aurc = pd.read_csv(OUT_DIR / "rq4_aurc_margin_benchmark.csv")
cad_aurc = cad_aurc[cad_aurc["dataset"] == "CADEC"].copy()
qa_aurc = pd.read_csv(OUT_DIR / "rq4_aurc_margin_qa.csv")

assert set(MODEL_ORDER_8) <= set(mm_curves["model"])
assert set(MODEL_ORDER_8) <= set(cad_curves["model"])
assert set(CONCEPT_SIGNALS) <= set(mm_curves["signal"])
assert set(CONCEPT_SIGNALS) <= set(cad_curves["signal"])
assert set(QA_SIGNALS) <= set(qa_curves["signal"])

saved_paths = []


def save_six(fig, filename):
    targets = [PRESENT / filename, OUT_DIR / filename, ALL_FIG / filename.replace("rq4_", "RQ4_")]
    for path in targets:
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    saved_paths.append(targets[0])
    print("Saved", targets[0])


def plot_rc(curves, models, signals, title, filename, footnote=None, small_models=None,
            n_override=None, ncols=None, nrows=None):
    small_models = small_models or set()
    n_models = len(models)
    if ncols is None:
        ncols, nrows = (4, 2) if n_models == 8 else (5, 1)
    fig_w = 4.15 * ncols
    fig_h = 3.85 * nrows + 1.35
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, model in zip(axes, models):
        sub = lookup_rows(curves, model)
        n_drawn = 0
        for sig in signals:
            s = sub[sub["signal"] == sig].sort_values("coverage")
            if s.empty:
                continue
            ls, color = STYLES[sig]
            ax.plot(s["coverage"], s["risk"], linestyle=ls, marker="o",
                    markersize=3, linewidth=1.55, color=color)
            n_drawn += 1
        n = n_override
        if n is None and "n" in sub.columns and len(sub) and sub["n"].notna().any():
            n = int(sub["n"].dropna().iloc[0])
        short = SHORT.get(model, model)
        flag = f" (n={n})" if n is not None else ""
        if model in small_models or (n is not None and n < 250):
            flag = f" (n={n}, SMALL)" if n is not None else " (SMALL)"
        ax.set_title(f"{short}{flag}", fontsize=10, pad=6)
        ax.set_xlabel("Coverage")
        ax.set_ylabel("Risk")
        ax.set_xlim(0.08, 1.02)
        ax.set_ylim(0.0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelbottom=True)
        if n_models == 8 and model in MODEL_ORDER_8[:3] and n_drawn != 6:
            print(f"WARNING: {title}: {model} drew {n_drawn}/6 series")
        if n_models == 5 and n_drawn != 4:
            print(f"WARNING: {title}: {model} drew {n_drawn}/4 series")
    for ax in axes[n_models:]:
        ax.axis("off")
    handles = [
        Line2D([0], [0], color=STYLES[s][1], linestyle=STYLES[s][0],
               marker="o", markersize=4, linewidth=1.6, label=signal_label(s))
        for s in signals
    ]
    fig.legend(handles, [h.get_label() for h in handles], loc="upper center",
               ncol=3 if len(signals) == 6 else 4, frameon=False,
               bbox_to_anchor=(0.5, 0.98), fontsize=9)
    fig.suptitle(title + "\nrisk = 1 − selective accuracy; lower is better",
                 y=1.02, fontsize=12)
    fig.subplots_adjust(top=0.82 if nrows == 2 else 0.72, bottom=0.14 if footnote else 0.08,
                        left=0.05, right=0.99, wspace=0.22, hspace=0.38)
    if footnote:
        fig.text(0.5, 0.01, footnote, ha="center", va="bottom", fontsize=8.0, color="#333333")
    save_six(fig, filename)


def plot_aurc(panel_specs, filename, suptitle, footnote, signals):
    fig, axes = plt.subplots(2, 1, figsize=(14.4, 9.8), sharey=True)
    n_sig = len(signals)
    w = 0.11 if n_sig == 6 else 0.16
    for ax, (models, aurc_df, panel_title, small_n_models) in zip(axes, panel_specs):
        x = np.arange(len(models))
        offsets = (np.arange(n_sig) - (n_sig - 1) / 2.0) * w
        for off, sig in zip(offsets, signals):
            vals = []
            for m in models:
                hit = lookup_rows(aurc_df, m)
                hit = hit[hit["signal"] == sig]
                vals.append(float(hit["AURC"].iloc[0]) if len(hit) else np.nan)
            ax.bar(x + off, vals, width=w * 0.90, color=STYLES[sig][1],
                   zorder=3, edgecolor="none")
        labels = []
        for m in models:
            lab = SHORT.get(m, m)
            n_hit = lookup_rows(aurc_df, m)["n"].dropna() if "n" in aurc_df.columns else pd.Series(dtype=float)
            n = int(n_hit.iloc[0]) if len(n_hit) else None
            if m in small_n_models or (n is not None and n < 250):
                lab = f"{lab}\nn={n}, SMALL" if n is not None else f"{lab}\nSMALL"
            elif n is not None:
                lab = f"{lab}\nn={n}"
            labels.append(lab)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8.5)
        ax.set_ylabel("AURC")
        ax.set_ylim(0.0, 1.02)
        ax.set_title(panel_title, loc="left", fontsize=11)
        ax.grid(True, axis="y", alpha=0.35, zorder=0)
        ax.set_axisbelow(True)
    handles = [Patch(facecolor=STYLES[s][1], label=signal_label(s)) for s in signals]
    fig.legend(handles=handles, loc="upper center", ncol=3 if n_sig == 6 else 4,
               frameon=False, bbox_to_anchor=(0.5, 1.02), fontsize=9)
    fig.suptitle(suptitle, y=1.045, fontsize=13)
    fig.text(0.5, 0.01, footnote, ha="center", va="bottom", fontsize=8.0, color="#333333")
    fig.tight_layout(rect=(0, 0.06, 1, 0.95))
    save_six(fig, filename)


plot_rc(
    mm_curves, MODEL_ORDER_8, CONCEPT_SIGNALS,
    "MedMentions — concept-level risk–coverage (all 8 models)",
    "rq4_risk_coverage_medmentions.png",
    footnote=MM_ENCODER_FOOTNOTE,
)
plot_rc(
    cad_curves, MODEL_ORDER_8, CONCEPT_SIGNALS,
    "CADEC — concept-level risk–coverage (all 8 models)",
    "rq4_risk_coverage_cadec.png",
    footnote=(
        "CADEC is patient-generated text. OpenBioLLM shown on n=233 defined-margin rows "
        "(SMALL); margin not imputed."
    ),
    small_models={"Llama3-OpenBioLLM-8B"},
)
plot_rc(
    qa_curves[qa_curves["dataset"] == "BioASQ"], MODEL_ORDER_QA, QA_SIGNALS,
    "BioASQ — answer-level risk–coverage (5 generatives, n=392)",
    "rq4_risk_coverage_bioasq.png",
    footnote=QA_FOOTNOTE,
    n_override=392,
)
plot_rc(
    qa_curves[qa_curves["dataset"] == "SQuAD2"], MODEL_ORDER_QA, QA_SIGNALS,
    "SQuAD — answer-level risk–coverage (5 generatives, n=139, SMALL)",
    "rq4_risk_coverage_squad.png",
    footnote=QA_FOOTNOTE,
    n_override=139,
    small_models=set(MODEL_ORDER_QA),
)

plot_aurc(
    [
        (MODEL_ORDER_8, mm_aurc, "MedMentions — all 8 models", set()),
        (MODEL_ORDER_8, cad_aurc, "CADEC — all 8 models", {"Llama3-OpenBioLLM-8B"}),
    ],
    "rq4_aurc_concept_medmentions_cadec.png",
    "RQ4 AURC — concept lane (lower is better)",
    MM_ENCODER_FOOTNOTE + " Concept-lane and QA-lane AURC are not comparable.",
    CONCEPT_SIGNALS,
)
plot_aurc(
    [
        (MODEL_ORDER_QA, qa_aurc[qa_aurc["dataset"] == "BioASQ"],
         "BioASQ — 5 generatives (n=392)", set()),
        (MODEL_ORDER_QA, qa_aurc[qa_aurc["dataset"] == "SQuAD2"],
         "SQuAD — 5 generatives (n=139, SMALL)", set(MODEL_ORDER_QA)),
    ],
    "rq4_aurc_qa_bioasq_squad.png",
    "RQ4 AURC — QA lane (lower is better)",
    "confidence = sequence log-probability; no UMLS margin in the QA lane; not "
    "comparable to concept-lane AURC.",
    QA_SIGNALS,
)

hashes_after = {p.name: file_sha256(p) for p in CSV_PATHS}
assert hashes_before == hashes_after, "Score CSVs were modified — abort"

print("\n=== 6 RQ4 figure paths ===")
for p in saved_paths:
    print(p)
assert len(saved_paths) == 6
print("CSV SHA256 unchanged. Display labels only; no AURC/curve recompute.")


## Task 2 — H=0 centrepiece (CADEC encoders)

When normalised entropy is exactly 0, UMLS `margin_mean` can still spread.
Encoders only (BERT-base, BioBERT, PubMedBERT). No imputed margins.


In [ ]:
# DISPLAY ONLY. Do not recompute RQ4 scores. Do not write CSVs.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / "outputs" / "rq4"
ALL_FIG = PROJECT_ROOT / "outputs" / "figures" / "all_rq_figures"
ALL_FIG.mkdir(exist_ok=True)

margin_path = PROJECT_ROOT / "outputs" / "rq3" / "umls_candidate_margin_cadec.csv"
entropy_path = PROJECT_ROOT / "outputs" / "rq3" / "entropy_cadec.csv"

margin = pd.read_csv(margin_path)
print("umls_candidate_margin_cadec.csv columns:", list(margin.columns))
entropy = pd.read_csv(entropy_path)
print("entropy_cadec.csv columns:", list(entropy.columns))

join_keys = ["instance_id", "model_name"]
for col in join_keys + ["normalised_entropy", "margin_mean"]:
    assert col in margin.columns, f"missing in margin CSV: {col}"
for col in join_keys + ["normalised_entropy"]:
    assert col in entropy.columns, f"missing in entropy CSV: {col}"

if "normalised_entropy" in margin.columns and margin["normalised_entropy"].notna().any():
    print("normalised_entropy already present in margin CSV — no merge.")
    df = margin.copy()
else:
    print("Merging margin + entropy on", join_keys)
    df = margin.merge(
        entropy[join_keys + ["normalised_entropy"]],
        on=join_keys, how="left", suffixes=("", "_from_entropy"),
    )

ENCODERS = ["BERT-base", "BioBERT", "PubMedBERT"]
assert "model_name" in df.columns
sub = df[df["model_name"].isin(ENCODERS)].copy()
sub["normalised_entropy"] = pd.to_numeric(sub["normalised_entropy"], errors="coerce")
sub["margin_mean"] = pd.to_numeric(sub["margin_mean"], errors="coerce")

h0 = sub[sub["normalised_entropy"] == 0].copy()
n_h0 = int(len(h0))
n_dropped = int(h0["margin_mean"].isna().sum())
print(f"undefined margin_mean dropped: {n_dropped}")
h0 = h0.dropna(subset=["margin_mean"])
zc = h0  # encoder-only, normalised_entropy==0 subset
n_used = int(len(zc))
std_m = float(zc["margin_mean"].std(ddof=1))
print(f"total zero-entropy encoder rows: {n_h0}")
print(f"N after dropping undefined margin_mean: {n_used}")
print(f"margin_mean std: {std_m:.6f}")

EXPECT_N, EXPECT_STD = 9875, 0.057
if abs(n_used - EXPECT_N) > 50 or abs(std_m - EXPECT_STD) > 0.01:
    print("STOP: N or std differs materially from expected ~9,875 / ≈0.057.")
    print(f"actual N={n_used} std={std_m}")
else:
    rng = np.random.default_rng(0)
    order = ENCODERS
    data = [zc.loc[zc["model_name"] == m, "margin_mean"].to_numpy() for m in order]
    fig, ax = plt.subplots(figsize=(8.4, 6.1))
    parts = ax.violinplot(
        data, positions=np.arange(len(order)),
        showmeans=True, showextrema=False, widths=0.72,
    )
    for pc in parts["bodies"]:
        pc.set_facecolor("#E69F00")
        pc.set_alpha(0.32)
        pc.set_edgecolor("#8B6914")
    if "cmeans" in parts:
        parts["cmeans"].set_color("#333333")
        parts["cmeans"].set_linewidth(1.1)
    for i, y in enumerate(data):
        x = i + rng.uniform(-0.14, 0.14, size=len(y))
        ax.scatter(x, y, s=7, alpha=0.16, c="#C47A00", linewidths=0, rasterized=True, zorder=3)
    ax.set_xticks(np.arange(len(order)))
    ax.set_xticklabels(order)
    ax.set_ylabel("margin_mean  (s(1) − s(2) at CUI level)")
    ax.set_xlabel("Encoder")
    ax.set_title("CADEC encoders — entropy = 0 for all instances, yet UMLS margin still spreads")
    ax.annotate(
        f"entropy = 0 for all {n_used} rows; margin_mean std = {std_m:.3f}",
        xy=(0.5, 0.97), xycoords="axes fraction", ha="center", va="top", fontsize=10,
    )
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    # Rare margin=1.0 outliers (s2=0) stretch the axis and squash the meaningful 0–0.3 spread.
    ax.set_ylim(-0.2, 0.4)
    n_tail = int((zc["margin_mean"] > 0.4).sum())
    ax.text(0.99, 0.02,
            f"y clipped to [-0.2, 0.4]; {n_tail} instances with margin > 0.4 (up to 1.0) not shown",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=8, color="0.45")
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.text(
        0.5, 0.015,
        "Negative margins occur when the five-rule disambiguation overrides the raw-cosine argmax; "
        "they are valid, not errors.",
        ha="center", va="bottom", fontsize=8.5, color="#333333",
    )
    out = OUT_DIR / "rq4_h0_centrepiece_cadec_encoders.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    fig.savefig(ALL_FIG / "RQ4_h0_centrepiece_cadec_encoders.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("Saved", out)
    print(f"TASK 2 annotation: entropy = 0 for all {n_used} rows; margin_mean std = {std_m:.3f}")
    print(f"TASK 2 n_tail (margin_mean > 0.4): {n_tail}")


## RQ4 — Spearman independence (UMLS margin vs entropy / confidence)

Lead evidence that margin is a new reliability axis. Loaded from
`outputs/rq4/rq4_spearman_margin.csv` only (source CSV is not overwritten).
MedMentions encoder margin–confidence is undefined (confidence is constant 1.0).


In [ ]:
# DISPLAY ONLY. Do not overwrite rq4_spearman_margin.csv. Do not recompute scores.
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / "outputs" / "rq4"
SIGNAL_LABELS = {
    "entropy":     "Semantic entropy",
    "confidence":  "Mapping confidence",
    "margin":      "UMLS margin",
    "random":      "Random (no ranking)",
    "combined_2":  "Entropy, then confidence if tied",
    "combined_3":  "Mean rank of entropy + confidence + margin",
}
SRC = OUT_DIR / "rq4_spearman_margin.csv"

def file_sha256(path):
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return h.hexdigest()

src_hash_before = file_sha256(SRC)

raw = pd.read_csv(SRC)
print("df.columns:", list(raw.columns))
print("df.head():")
print(raw.head().to_string(index=False))

COL_DS, COL_MODEL = "dataset", "model"
COL_RHO_H, COL_RHO_C = "spearman_margin_entropy", "spearman_margin_confidence"
COL_P_H, COL_P_C = "p_margin_entropy", "p_margin_confidence"
for col in [COL_DS, COL_MODEL, COL_RHO_H, COL_RHO_C, COL_P_H, COL_P_C, "n"]:
    assert col in raw.columns, f"missing column: {col}"
print("Mapped: model -> model; dataset -> dataset; "
      "rho(margin,entropy) -> spearman_margin_entropy; "
      "rho(margin,confidence) -> spearman_margin_confidence; "
      "p-values -> p_margin_entropy, p_margin_confidence")

MODEL_LABEL = {
    "BERT-base": "BERT-base",
    "BioBERT": "BioBERT",
    "PubMedBERT": "PubMedBERT",
    "FLAN-T5-base": "FLAN-T5",
    "BioMistral-7B": "BioMistral-7B",
    "Mistral-7B-Instruct-v0.1": "Mistral-7B",
    "Llama3-OpenBioLLM-8B": "OpenBioLLM-8B",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
}
ENCODERS = ["BERT-base", "BioBERT", "PubMedBERT"]
GENERATIVES = [
    "FLAN-T5-base", "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
]
DATASETS = ["MedMentions", "CADEC"]
# Grouping: family first (encoders, then generatives), within family by dataset
# (MedMentions, then CADEC), models in the lists above. Clusters the MM-encoder
# n/a block and the MM-generative independence-claim block.
print("Row grouping: CADEC encoders, then MedMentions generatives, then CADEC generatives.")
print("MedMentions encoder rows are OMITTED (not plotted as n/a).")

work = raw.copy()
work[COL_RHO_H] = pd.to_numeric(work[COL_RHO_H], errors="coerce")
work[COL_RHO_C] = pd.to_numeric(work[COL_RHO_C], errors="coerce")
work[COL_P_H] = pd.to_numeric(work[COL_P_H], errors="coerce")
work[COL_P_C] = pd.to_numeric(work[COL_P_C], errors="coerce")

lookup = work.set_index([COL_DS, COL_MODEL])
degenerate = []  # list of (dataset, model, which_rho, reason)

rows = []
for family_name, models in [("encoder", ENCODERS), ("generative", GENERATIVES)]:
    for ds in DATASETS:
        for model in models:
            key = (ds, model)
            if key in lookup.index:
                hit = lookup.loc[key]
                if isinstance(hit, pd.DataFrame):
                    hit = hit.iloc[0]
                rho_h = float(hit[COL_RHO_H]) if pd.notna(hit[COL_RHO_H]) else np.nan
                rho_c = float(hit[COL_RHO_C]) if pd.notna(hit[COL_RHO_C]) else np.nan
                p_h = float(hit[COL_P_H]) if pd.notna(hit[COL_P_H]) else np.nan
                p_c = float(hit[COL_P_C]) if pd.notna(hit[COL_P_C]) else np.nan
                n_val = int(hit["n"]) if pd.notna(hit["n"]) else np.nan
            else:
                rho_h = rho_c = p_h = p_c = np.nan
                n_val = np.nan

            # MedMentions encoders: omit entirely (confidence constant; margin = mention-span).
            if ds == "MedMentions" and model in ENCODERS:
                print(f"OMIT {ds} {MODEL_LABEL[model]} (not shown; correlation undefined / not independence axis)")
                continue

            if pd.isna(rho_h):
                degenerate.append((ds, MODEL_LABEL[model], "rho(margin, entropy)", "NaN in source CSV"))
            if pd.isna(rho_c):
                degenerate.append((ds, MODEL_LABEL[model], "rho(margin, confidence)",
                                   "NaN in source CSV or zero variance"))

            rows.append({
                "model": MODEL_LABEL[model],
                "dataset": ds,
                "family": family_name,
                "rho_margin_entropy": None if pd.isna(rho_h) else round(rho_h, 2),
                "rho_margin_confidence": None if pd.isna(rho_c) else round(rho_c, 2),
                "p_margin_entropy": None if pd.isna(p_h) else float(p_h),
                "p_margin_confidence": None if pd.isna(p_c) else float(p_c),
                "n": None if pd.isna(n_val) else int(n_val),
            })

tidy = pd.DataFrame(rows)
assert len(tidy) == 13, f"expected 13 rows (3 CADEC encoders + 5 MM gen + 5 CADEC gen), got {len(tidy)}"
na_cells = tidy[["rho_margin_entropy", "rho_margin_confidence"]].isna().sum().sum()
assert na_cells == 0, f"n/a cells remain after dropping MM encoders: {na_cells}"
print(f"Rows plotted: {len(tidy)}; n/a cells remaining: {int(na_cells)}")
print("\n=== Degenerate / n/a cells (must be none after dropping MM encoders) ===")
if not degenerate:
    print("(none)")
for ds, m, which, reason in degenerate:
    print(f"  {ds} | {m} | {which}: {reason}")

# --- verify MM-generative ranges from the rounded tidy values (not hardcoded) ---
mm_gen = tidy[(tidy["dataset"] == "MedMentions") & (tidy["family"] == "generative")]
mm_h = pd.to_numeric(mm_gen["rho_margin_entropy"], errors="coerce")
mm_c = pd.to_numeric(mm_gen["rho_margin_confidence"], errors="coerce")
print("\n=== MedMentions generative rho (2 dp, from data) ===")
print(mm_gen[["model", "rho_margin_entropy", "rho_margin_confidence"]].to_string(index=False))
print(f"rho(margin, entropy)    min={mm_h.min():.2f}  max={mm_h.max():.2f}")
print(f"rho(margin, confidence) min={mm_c.min():.2f}  max={mm_c.max():.2f}")
if not (mm_h.min() >= -0.15 and mm_h.max() <= 0.15):
    print("NOTE: MM-generative rho(margin, entropy) is not all near 0; using actual min/max above, not a forced ~0 claim.")
if not (mm_c.min() >= 0.20 and mm_c.max() <= 0.50):
    print("NOTE: MM-generative rho(margin, confidence) falls outside roughly 0.25–0.46; using actual min/max above.")

def fmt2(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    return f"{float(v):.2f}"

tidy_out = pd.DataFrame({
    "model": tidy["model"],
    "dataset": tidy["dataset"],
    "rho_margin_entropy": [fmt2(v) for v in tidy["rho_margin_entropy"]],
    "rho_margin_confidence": [fmt2(v) for v in tidy["rho_margin_confidence"]],
    "p_margin_entropy": tidy["p_margin_entropy"],
    "p_margin_confidence": tidy["p_margin_confidence"],
    "n": ["" if v is None or (isinstance(v, float) and np.isnan(v)) else int(v) for v in tidy["n"]],
})
out_csv = OUT_DIR / "rq4_spearman_independence.csv"
assert out_csv.resolve() != SRC.resolve(), "refusing to overwrite source Spearman CSV"
tidy_out.to_csv(out_csv, index=False)
print("Wrote", out_csv)
print(tidy_out.to_string(index=False))

NA = "n/a"
GREY = "#d9d9d9"
CAPTION = (
    "Independence is claimed for generative models. On MedMentions the margin "
    "is rank-uncorrelated with semantic entropy (rho ~ 0) and only weakly-to-moderately correlated "
    "with confidence. MedMentions encoders are omitted: their confidence is constant (correlation "
    "undefined) and their margin reflects input-mention difficulty, not model uncertainty. CADEC "
    "encoders (BERT/BioBERT/PubMedBERT) are shown and are moderately correlated — margin is not "
    "independent there."
)

def fmt_rho(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return NA
    return f"{v:.2f}"

# ----- OUTPUT A: matplotlib table -----
fig, ax = plt.subplots(figsize=(9.6, 6.4))
ax.axis("off")
col_labels = ["Model", "Dataset",
              f"ρ({SIGNAL_LABELS['margin']}, {SIGNAL_LABELS['entropy']})",
              f"ρ({SIGNAL_LABELS['margin']}, {SIGNAL_LABELS['confidence']})"]
cell_text = []
na_mask = []
for _, r in tidy.iterrows():
    te, tc = fmt_rho(r["rho_margin_entropy"]), fmt_rho(r["rho_margin_confidence"])
    cell_text.append([r["model"], r["dataset"], te, tc])
    na_mask.append([False, False, te == NA, tc == NA])
table = ax.table(
    cellText=cell_text, colLabels=col_labels, loc="upper center", cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.15, 1.35)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#cccccc")
    if row == 0:
        cell.set_facecolor("#4a4a4a")
        cell.set_text_props(color="white", weight="bold")
        continue
    if na_mask[row - 1][col]:
        cell.set_facecolor(GREY)
        cell.set_text_props(color="#555555")
    elif row % 2 == 0:
        cell.set_facecolor("#f7f7f7")
    else:
        cell.set_facecolor("white")
ax.set_title("RQ4 — UMLS margin vs existing signals (Spearman rank correlation)", pad=12, fontsize=12)
fig.tight_layout(rect=(0, 0.16, 1, 0.98))
fig.text(0.5, 0.02, CAPTION, ha="center", va="bottom", fontsize=8.0, color="#333333", wrap=True)
table_png = OUT_DIR / "rq4_spearman_independence_table.png"
fig.savefig(table_png, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved", table_png)

# ----- OUTPUT B: heatmap -----
n_rows = len(tidy)
mat = np.full((n_rows, 2), np.nan, dtype=float)
for i, r in tidy.iterrows():
    ii = int(i)
    if r["rho_margin_entropy"] is not None:
        mat[ii, 0] = r["rho_margin_entropy"]
    if r["rho_margin_confidence"] is not None:
        mat[ii, 1] = r["rho_margin_confidence"]
masked = np.ma.masked_invalid(mat)
cmap = plt.colormaps["RdBu_r"].copy()
cmap.set_bad(GREY)

fig, ax = plt.subplots(figsize=(8.4, 7.6))
im = ax.imshow(masked, cmap=cmap, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks([0, 1])
ax.set_xticklabels(
    [f"ρ({SIGNAL_LABELS['margin']}, {SIGNAL_LABELS['entropy']})",
     f"ρ({SIGNAL_LABELS['margin']}, {SIGNAL_LABELS['confidence']})"],
    fontsize=8.5,
)
ax.set_yticks(np.arange(n_rows))
ax.set_yticklabels([f"{r.model}  ·  {r.dataset}" for r in tidy.itertuples()], fontsize=9)
ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
for i in range(n_rows):
    for j in range(2):
        if masked.mask[i, j]:
            ax.text(j, i, NA, ha="center", va="center", fontsize=8.5, color="#555555")
        else:
            val = mat[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8.5,
                    color="white" if abs(val) >= 0.55 else "#111111")
# separators: after CADEC encoders (3), after MM generatives (3+5=8)
for y, lw, c in [(2.5, 1.3, "0.15"), (7.5, 0.6, "0.45")]:
    ax.axhline(y, color=c, lw=lw, clip_on=False)
ax.set_title("RQ4 — UMLS margin vs existing signals (Spearman rank correlation)",
             fontsize=12, pad=18)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label("Spearman ρ", fontsize=9)
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
fig.tight_layout(rect=(0, 0.18, 1, 0.98))
fig.text(0.5, 0.015, CAPTION, ha="center", va="bottom", fontsize=8.0, color="#333333", wrap=True)
heat_png = OUT_DIR / "rq4_spearman_independence.png"
fig.savefig(heat_png, dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved", heat_png)
present_fig = OUT_DIR / "present" / "figures"
present_fig.mkdir(parents=True, exist_ok=True)
for p in (table_png, heat_png):
    (present_fig / p.name).write_bytes(p.read_bytes())
    print("Copied", present_fig / p.name)


src_hash_after = file_sha256(SRC)
assert src_hash_before == src_hash_after, "Source Spearman CSV was modified — abort"
print("Source CSV SHA256 unchanged:", src_hash_after[:16] + "...")
print("SUMMARY: MM-encoder rows omitted; n/a cells remaining: 0; "
      f"MM-generative rho(margin,entropy) [{mm_h.min():.2f}, {mm_h.max():.2f}]; "
      f"rho(margin,confidence) [{mm_c.min():.2f}, {mm_c.max():.2f}]")
